# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List all record sets and their fields
print('Record sets and their field @ids:')
for record_set in metadata.record_sets:
    print(f"- Record set @id: {record_set['@id']}, name: {record_set.get('name', 'N/A')}")
    if 'field' in record_set:
        fields = record_set['field']
        # 'field' can be list or object
        if isinstance(fields, dict):
            fields = [fields]
        for field in fields:
            if isinstance(field, dict):
                print(f"    - Field @id: {field.get('@id', field)}, name: {field.get('name', 'N/A')}")
            else:
                print(f"    - Field @id: {field}")
    else:
        print("    - No fields found.")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Get all record set @ids
record_set_ids = [rs['@id'] for rs in metadata.record_sets]

dataframes = {}
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    dataframes[record_set_id] = pd.DataFrame(records)

# For demonstration, select the first record set.
if record_set_ids:
    demo_record_set_id = record_set_ids[0]
    print(f"Columns for {demo_record_set_id}: {dataframes[demo_record_set_id].columns.tolist()}")
    display(dataframes[demo_record_set_id].head())
else:
    print("No record sets available in metadata.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section should include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
# Example: Pick a representative numeric field for EDA. Adjust numeric_field_id/group_field_id as appropriate from the overview.
import numpy as np

# You should replace these IDs with the appropriate ones from your schema.
# For demonstration, let's search for fields that are likely numeric (e.g., PHQ-9, GAD-7 scores)
if record_set_ids:
    df = dataframes[demo_record_set_id]
    # Guessing a suitable numeric column
    candidate_numeric = [col for col in df.columns if 'score' in col.lower() or df[col].dtype in [np.int64, np.float64]]
    if candidate_numeric:
        numeric_field = candidate_numeric[0]
        threshold = df[numeric_field].mean(skipna=True)  # Use mean as threshold for demonstration
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
        display(filtered_df.head())

        # Normalize numeric_field
        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"Normalized {numeric_field} for filtered records:")
        display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Attempt grouping by a demographic field (e.g., gender, village, education)
        candidate_groups = [col for col in df.columns if col.lower() in ['gender', 'village', 'level_of_education']]
        if candidate_groups:
            group_field = candidate_groups[0]
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().to_frame()
            print(f"Grouped data by {group_field} (mean {numeric_field}):")
            display(grouped_df)
        else:
            print("No obvious demographic field found for grouping.")
    else:
        print("No numeric fields found for filtering/EDA.")
else:
    print("No record sets available for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot a histogram of the main numeric field and a boxplot by group, if possible.
if record_set_ids and candidate_numeric:
    plt.figure(figsize=(6,4))
    df[numeric_field].hist(bins=15)
    plt.title(f'Histogram of {numeric_field}')
    plt.xlabel(numeric_field)
    plt.ylabel('Count')
    plt.show()

    if candidate_groups:
        plt.figure(figsize=(6,4))
        sns.boxplot(x=group_field, y=numeric_field, data=df)
        plt.title(f'{numeric_field} by {group_field}')
        plt.show()
else:
    print("No suitable fields available for visualization.")

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- Successfully loaded the FAIR² Mental Health Survey dataset and reviewed its metadata structure with `mlcroissant`.
- Inspected available record sets, fields, and explored the loaded data using inferred field IDs.
- Demonstrated filtering, normalization, and simple grouping/aggregation for quantitative and demographic fields.
- Provided example plots for exploratory visualization.

For a thorough analysis, examine the data dictionary and metadata schema for precise field IDs and definitions applicable to your specific research focus.